In [ ]:
import networkx as nx
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib.patches as patches
import matplotlib.font_manager
from scipy.stats import pearsonr
from scipy.stats import linregress
from matplotlib import pyplot as plt
import matplotlib as mpl
from pycirclize import Circos
matplotlib.font_manager.fontManager.addfont('/h/tianyi/TS_datasets_reversion/Cell_Press_plot/Arial.ttf')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 8 
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'
plt.rcParams['axes.titlecolor'] = 'black'
plt.rcParams['legend.labelcolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.5

## Figure4A,B

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

primary_subnetwork_all=pd.read_csv('primary_subnetwork_all.txt',sep='\t')
tissue='Bowel'
gene='APC'
def subnet_filter(tissue,genes):
    primary_subnetwork_eg=primary_subnetwork_all[(primary_subnetwork_all['tissue']==tissue) & 
                                                (primary_subnetwork_all['seed']==gene)]
    primary_subnetwork_genelist=np.union1d(primary_subnetwork_eg['sub_nodes'],primary_subnetwork_eg['seed'])
    return primary_subnetwork_genelist

def draw_venn2(setA, setB, labels ,
               colors ,
               alpha, figsize, fontsize,filename):
    plt.figure(figsize=figsize)
    v = venn2([set(setA), set(setB)], set_labels=labels, set_colors=colors, alpha=alpha)
    for text in v.set_labels:
        text.set_fontsize(fontsize)
    for text in v.subset_labels:
        if text:  # 有些区域可能没有文本
            text.set_fontsize(fontsize)
    v.get_patch_by_id('11').set_color('#cc3c4c')
    v.get_patch_by_id('11').set_alpha(alpha)
    plt.savefig(filename,bbox_inches='tight',dpi=300)
draw_venn2(setA, setB, labels ,
               colors ,
               alpha, figsize, fontsize,filename)

lung_egfr=subnet_filter('Lung','EGFR')
cns_egfr=subnet_filter('CNS_Brain','EGFR')
setA=lung_egfr
setB=cns_egfr
labels=['Lung specific','CNS/Brain specific','Shared']
colors=['#ADCFA0','#3481B3',
'#CC3C4C']
alpha=0.6
figsize=(1.5,1.5)
fontsize=8
filename='EGFR_venn.pdf'

primary_subnetwork_all['tissue'].drop_duplicates()
setA=subnet_filter('Bowel','APC')
setB=subnet_filter('Uterus','APC')
setC=subnet_filter('Esophagus_Stomach','APC')
primary_subnetwork_all


set_labels=['Bowel','Uterus','Esophagus/Stomach']
from matplotlib_venn import venn3

plt.figure(figsize=(1.5,1.5))
v = venn3([set(setA), set(setB), set(setC)],alpha=0.6,set_labels=set_labels)
# 调整标签字体大小
for text in v.set_labels:
    text.set_fontsize(8)
# 调整交集区域数字字体大小
for text in v.subset_labels:
    if text:
        text.set_fontsize(8)

plt.savefig('APC_venn.pdf',bbox_inches='tight',dpi=300)


## Figure4C

In [ ]:
###subnetwork distance cluster and plot umap/DBSCAN


library(dbscan)
library(fpc)
library(plotly)
library(RColorBrewer)
library(tidyverse)
library(umap)
install.packages('dbscan')
install.packages('fpc')
install.packages('umap')

set.seed(123)

# loading subnetwork distance matrix
dis_tsne=data.frame(read.table('tsne_DBSCAN_cluster.txt',sep='\t',header=TRUE,comment.char = ''))
dat <- data.frame(read.table("distance_subnetwork_matrix.txt", sep = "\t", header=TRUE, row.names = 1,comment.char = ''))
subnetwork_dist_matrix <- as.matrix(dat)
rownames(subnetwork_dist_matrix) <- rownames(dat)
colnames(subnetwork_dist_matrix) <- rownames(dat)

# normalize distance
subnetwork_dist_matrix[row(subnetwork_dist_matrix)==col(subnetwork_dist_matrix)] <- 0

max_dist <- max(subnetwork_dist_matrix)
min_dist <- min(subnetwork_dist_matrix)

normalized_subnetwork_dist_matrix <- (subnetwork_dist_matrix-min_dist)/(max_dist-min_dist)
normalized_subnetwork_dist_matrix[row(normalized_subnetwork_dist_matrix)==col(normalized_subnetwork_dist_matrix)] <- 0


# UMAP
subnetwork.umap <- umap(normalized_subnetwork_dist_matrix, input="dist", min_dist=0.1, n_neighbors=15)
dot.positions <- tibble(x=subnetwork.umap$layout[, 1], y=subnetwork.umap$layout[, 2]) 

# DBSCAN clustering
db <- fpc::dbscan(dot.positions, eps = 1, MinPts = 5)

dot.positions$label <- rownames(dat)
dot.positions$cluster <- factor(db$cluster, levels=c(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19))

my.colors <- colorRampPalette(brewer.pal(8,"Set1"))(19)
my.colors <- c("#BEBEBE", my.colors)

dot.positions_df=data.frame(dot.positions)
colnames(dot.positions_df)=c('umap 1','umap 2','label','umap_DBSCAN_id')
dot.positions_df_color=merge(dot.positions_df,dis_tsne,by.x='label',by.y='subnetwork_id')

p <- ggplot(dot.positions, aes(x=x, y=y, color=cluster, label=label, shape=cluster))+
  geom_point(size=2)+
  #scale_shape_manual(values=c(2, rep(19, 19)))+
  #scale_color_manual(values = my.colors)+
  scale_color_brewer(palette="Set1")+
  theme_bw()

ggplotly(p)
unique(dot.positions$cluster)
dbscan::kNNdistplot(dot.positions, k =  5)
abline(h = 1, lty = 2)



tsne_plot=data.frame(read.table('G:/课题/tissue_specificity_manuscript/素材/network_distance_umap/tsne_output_plot_color_cluster7_perplexity10.txt',sep='\t',header=1,comment.char = ""))

tsne_plot=data.frame(read.table('G:/课题/tissue_specificity_manuscript/素材/network_distance_umap/tsne_DBSCAN_cluster.txt',sep='\t',header=1,comment.char = ""))
tsne_plot$subnetwork_id_process=paste(tsne_plot$tissue_name,':',tsne_plot$seed,sep='')
tsne_plot$subnetwork_id_process=tsne_plot$subnetwork_id
tsne_plot=dot.positions_df_color[c('umap 1','umap 2','umap_DBSCAN_id','tissue','seed','tissue_cor_v2','label')]
tsne_plot$plot_label=''
tsne_plot[which(tsne_plot$label %in% c('Adrenal gland-FGFR1','Biliary tract-TP53',
                                               'Bladder/Urinary tract-RXRA',
                                               'Bladder/Urinary tract-PIK3CA',
                                               'Bone-IDH1',
                                               'Bowel-APC',
                                               'Breast-ERBB2',
                                               'Breast-PIK3CA',
                                               'CNS/Brain-EGFR',
                                               'Esophagus/Stomach-CDH1',
                                               'Head and neck-NOTCH1',
                                               'Head and neck-PIK3CA',
                                               'Kidney-VHL',
                                               'Liver-CTNNB1','Liver-DMD','Lung-EGFR','Lung-BRAF','Lung-KRAS','Lymphoid-MYD88',
                                               'Myeloid-FLT3','Myeloid-JAK2','Myeloid-KIT','Ovary/Fallopian tube-TP53','Pancreas-KRAS','Pancreas-TP53',
                                               'Prostate-SPOP',
                                               'Skin-BRAF',
                                               'Skin-NRAS',
                                               'Skin-KIT','Thyroid-BRAF','Uterus-PIK3CA')),]$plot_label=tsne_plot[which(tsne_plot$label %in%c('Adrenal gland-FGFR1','Biliary tract-TP53',
                                                                                                                                                 'Bladder/Urinary tract-RXRA',
                                                                                                                                                 'Bladder/Urinary tract-PIK3CA',
                                                                                                                                                 'Bone-IDH1',
                                                                                                                                                 'Bowel-APC',
                                                                                                                                                 'Breast-ERBB2',
                                                                                                                                                 'Breast-PIK3CA',
                                                                                                                                                 'CNS/Brain-EGFR',
                                                                                                                                                 'Esophagus/Stomach-CDH1',
                                                                                                                                                 'Head and neck-NOTCH1',
                                                                                                                                                 'Head and neck-PIK3CA',
                                                                                                                                                 'Kidney-VHL',
                                                                                                                                                 'Liver-CTNNB1','Liver-DMD','Lung-EGFR','Lung-BRAF','Lung-KRAS','Lymphoid-MYD88',
                                                                                                                                                 'Myeloid-FLT3','Myeloid-JAK2','Myeloid-KIT','Ovary/Fallopian tube-TP53','Pancreas-KRAS','Pancreas-TP53',
                                                                                                                                                 'Prostate-SPOP',
                                                                                                                                                 'Skin-BRAF',
                                                                                                                                                 'Skin-NRAS',
                                                                                                                                                 'Skin-KIT','Thyroid-BRAF','Uterus-PIK3CA')),]$label



write.table(tsne_plot,'tsne_output_plot_color_umap7_DBSCAN.txt',sep='\t',row.names = F)
# 创建基础图表
library(ggthemes)
setwd('G:\\课题\\tissue_specificity_manuscript\\素材\\network_distance_umap')
p <- ggscatter(tsne_plot, x='umap 1', y='umap 2', color = 'tissue_cor_v2',
               palette = unique(tsne_plot[c('tissue','tissue_cor_v2')])$tissue_cor_v2,label=tsne_plot$plot_label,
               font.label = c(8, "plain"), show.legend = FALSE,repel = T)+theme(legend.position = "none")+
  theme(
    legend.position = "none",panel.border = element_blank(),
    axis.line.y = element_line(size = (0.5/1.07)*0.5),
    axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
    axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
    axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    axis.line = element_line(colour = "black"),
    panel.background = element_rect(fill = "transparent", color = NA, size = (0.5/1.07)*0.5),
    axis.text = element_text(size = 8),
    axis.title = element_text(size = 8),
    legend.text = element_text(size = 8),  # Modify legend labels font and size
    legend.title = element_text(size = 8),  # Modify legend title font and size
    plot.title = element_text(size = 8, hjust = 0.5),
    axis.text.x = element_text(size = 8, family = "sans", color = "black"),
    axis.text.y = element_text(size = 8, family = "sans", color = "black")
  ) 
ggsave("tsne_output_plot_color_umap7_DBSCAN.pdf", p, width = 8, height = 8.5/2)

custom_colors <- c('#e6194B', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
                   '#911eb4', '#42d4f4')
tsne_plot$umap_DBSCAN_id<- factor(tsne_plot$umap_DBSCAN_id)
p <- ggscatter(tsne_plot, x='umap 1', y='umap 2', 
               color = 'umap_DBSCAN_id',palette = custom_colors)
ggsave("umap8_DBSCAN_cluster_number.pdf", p, width = 8, height = 8.5/2)




In [ ]:

each_pathway_df_all=pd.read_csv('/C2_CP_GSEA_pathway.txt',sep='\t')
GSEA_kegg=each_pathway_df_all[each_pathway_df_all['pathway_id'].str.contains('REACTOME')]


def gsea_enrichment(gene_list,each_pathway_df_all,background_network_gene_entrez):
    each_pathway_enrichment_all=pd.DataFrame()
    for pos in range(each_pathway_df_all.shape[0]):
        each_pathway_df_eg=each_pathway_df_all.iloc[pos:(pos+1),:]
        each_pathway_df_eg_gene=pd.Series(each_pathway_df_eg['pathway_genes'].str.split('|').iloc[0]).astype(int).tolist()
        inlist_inpath=np.intersect1d(gene_list,each_pathway_df_eg_gene)
        outlist_inpath=np.intersect1d(np.setdiff1d(background_network_gene_entrez['Entrez_id'],gene_list),each_pathway_df_eg_gene)
        inlist_outpath=np.intersect1d(gene_list,np.setdiff1d(background_network_gene_entrez['Entrez_id'],each_pathway_df_eg_gene))
        outlist_outpath=np.intersect1d(np.setdiff1d(background_network_gene_entrez['Entrez_id'],gene_list),np.setdiff1d(background_network_gene_entrez['Entrez_id'],each_pathway_df_eg_gene))
        inlist_inpath_gene_name='|'.join(background_network_gene_entrez[background_network_gene_entrez['Entrez_id'].isin(inlist_inpath)]['all_gene_symbol'].astype(str).tolist())
        enrich_fisher_od,enrich_fisher_p=fisher_exact([[len(inlist_inpath), len(outlist_inpath)], [len(inlist_outpath), len(outlist_outpath)]])
        each_pathway_enrichment=pd.DataFrame()
        each_pathway_enrichment['pathway_id']=[''.join(each_pathway_df_eg['pathway_id'])]
        each_pathway_enrichment['overlap_gene_name']=[inlist_inpath_gene_name]
        each_pathway_enrichment['gene_list']=['|'.join(pd.Series(gene_list).astype(str))]
        each_pathway_enrichment['inlist_inpath']=[len(inlist_inpath)]
        each_pathway_enrichment['outlist_inpath']=[len(outlist_inpath)]
        each_pathway_enrichment['inlist_outpath']=[len(inlist_outpath)]
        each_pathway_enrichment['outlist_outpath']=[len(outlist_outpath)]
        each_pathway_enrichment['enrich_fisher_od']=[enrich_fisher_od]
        each_pathway_enrichment['enrich_fisher_p']=[enrich_fisher_p]
        each_pathway_enrichment_all=pd.concat([each_pathway_enrichment,each_pathway_enrichment_all])
    return(each_pathway_enrichment_all)

def gene_list_enrichment_main(seed_gene,subnetwork_stage,stage,tissue):
    subnetwork_stage_seed=subnetwork_stage[(subnetwork_stage['seed']==seed_gene) & (subnetwork_stage['tissue']==tissue)]
    gene_list_eg=np.union1d(subnetwork_stage_seed['sub_nodes'],subnetwork_stage_seed['seed'])
    gene_list_entrez_eg=background_network_gene_entrez[background_network_gene_entrez['all_gene_symbol'].isin(gene_list_eg)]['Entrez_id'].astype(int).tolist()
    subnetwork_tissue_stage_enrich=gsea_enrichment(gene_list_entrez_eg,GSEA_kegg,background_network_gene_entrez)
    subnetwork_tissue_stage_enrich['pathway_id_name']=subnetwork_tissue_stage_enrich['pathway_id'].str.replace('KEGG_','').str.replace('_',' ').str.capitalize().tolist()
    subnetwork_tissue_stage_enrich['tissue']=tissue
    subnetwork_tissue_stage_enrich['stage']=stage
    subnetwork_tissue_stage_enrich['seed_gene']=seed_gene
    subnetwork_tissue_stage_enrich['FDR_BH']=multitest.multipletests(subnetwork_tissue_stage_enrich['enrich_fisher_p'], method='fdr_bh')[1]
    subnetwork_tissue_stage_enrich['Subnetwork ID']=subnetwork_tissue_stage_enrich['tissue']+'-'+subnetwork_tissue_stage_enrich['seed_gene']
    subnetwork_tissue_stage_enrich['Gene ratio']=subnetwork_tissue_stage_enrich['inlist_inpath']/(subnetwork_tissue_stage_enrich['inlist_inpath']+subnetwork_tissue_stage_enrich['outlist_inpath'])
    subnetwork_tissue_stage_enrich['Pathway name']=subnetwork_tissue_stage_enrich['pathway_id'].str.replace('REACTOME_','').str.replace('_',' ').str.capitalize().tolist()
    subnetwork_tissue_stage_enrich['P-values']=subnetwork_tissue_stage_enrich['enrich_fisher_p']
    subnetwork_tissue_stage_enrich['False discovery rate']=subnetwork_tissue_stage_enrich['FDR_BH']
    subnetwork_tissue_stage_enrich['Overlapping gene sets']=subnetwork_tissue_stage_enrich['overlap_gene_name']
    subnetwork_tissue_stage_enrich_excel=subnetwork_tissue_stage_enrich[['Subnetwork ID','Pathway name','Gene ratio',
                                                                        'Overlapping gene sets','P-values','False discovery rate']]
    subnetwork_tissue_stage_enrich_excel_sig=subnetwork_tissue_stage_enrich_excel[subnetwork_tissue_stage_enrich_excel['False discovery rate']<0.05]
    subnetwork_tissue_stage_enrich_excel_sig.to_excel(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/supplyment_figure/subnetwork_enrichment_pathway/Subnetwork_enrichment_geneset/{stage}_{tissue}_{seed_gene}_reactome_pathway.xlsx',index=False)
    return(subnetwork_tissue_stage_enrich_excel_sig)

####primary enrichment
from scipy.stats import fisher_exact
from statsmodels.stats import multitest
primary_subnetwork=pd.read_csv('primary_subnetwork_all.txt',sep='\t')
primary_seed_tissue=primary_subnetwork[['seed','tissue']].drop_duplicates()
background_network=pd.read_csv('human_ppi_v03.tsv',sep='\t',header=None)
gene_entrez=pd.read_csv('entrez_id_symbol_all.txt',sep='\t')
background_network_gene=pd.DataFrame(np.union1d(background_network[0],background_network[1]))
background_network_gene.columns=['subnetwork_background']
background_network_gene_entrez=pd.merge(background_network_gene,gene_entrez,left_on='subnetwork_background',right_on='all_gene_symbol')

for pos_eg in range(0,primary_seed_tissue.shape[0]):
    primary_seed_tissue_eg=primary_seed_tissue.iloc[pos_eg:(pos_eg+1),:]
    seed_gene_eg=''.join(primary_seed_tissue_eg['seed'])
    stage_eg='Primary'
    tissue_eg=''.join(primary_seed_tissue_eg['tissue'])
    gsea_enrichment_tissue_eg=gene_list_enrichment_main(seed_gene_eg,primary_subnetwork,stage_eg,tissue_eg)

def gene_list_enrichment_main_pool(pos_eg):
    primary_seed_tissue_eg=primary_seed_tissue.iloc[pos_eg:(pos_eg+1),:]
    seed_gene_eg=''.join(primary_seed_tissue_eg['seed'])
    stage_eg='Primary'
    tissue_eg=''.join(primary_seed_tissue_eg['tissue'])
    gsea_enrichment_tissue_eg=gene_list_enrichment_main(seed_gene_eg,primary_subnetwork,stage_eg,tissue_eg)
from multiprocessing import Pool, cpu_count
pool = Pool(40)
rl =pool.map(gene_list_enrichment_main_pool, range(0,primary_seed_tissue.shape[0])) 
pool.close()#关闭进程池，不再接受新的进程
pool.join()#主进程阻塞等待子进程的退出

enrich_file_all=os.listdir('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/supplyment_figure/subnetwork_enrichment_pathway/Subnetwork_enrichment_geneset')
enrich_path_all=pd.DataFrame()
for enrich_file_eg in enrich_file_all:
    enrich_file_eg_read=pd.read_excel(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/supplyment_figure/subnetwork_enrichment_pathway/Subnetwork_enrichment_geneset/{enrich_file_eg}')
    enrich_path_all=pd.concat([enrich_file_eg_read,enrich_path_all])
enrich_path_all.to_excel(f'/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/supplyment_figure/subnetwork_enrichment_pathway/Subnetwork enrichment p value_426.xlsx',index=False)

In [ ]:
enrich_p_value=pd.read_csv('gsea_reactome_enrichment_tissue_primary_all_v2.txt',sep='\t')
primary_subnet_count=pd.read_csv('top20_pathway_share.txt',sep='\t')

library(ggpubr)
library("ggsci")
library(ggplot2)
library(ggsignif)
library(ggpubr)
library(ComplexHeatmap)
library(circlize)


########################specific pathway#########################
enrich_p_value=data.frame(read.table('gsea_reactome_enrichment_tissue_primary_all_v2.txt',header = T,sep='\t', row.names = 1))
primary_subnet_count=data.frame(read.table('top20_pathway_share.txt',sep='\t',header = T, row.names = 1))
enrich_p_value_share=enrich_p_value[which(rownames(enrich_p_value) %in% rownames(primary_subnet_count)),]
enrich_p_value_log10=-log10(as.matrix(enrich_p_value_share))

tissue_anno1=gsub('Primary_','',colnames(enrich_p_value_log10))
tissue_anno2=lapply(tissue_anno1, 
                    function(x) {
                      str1_pro=strsplit(x, "_")[[1]]
                      str2_pro=head(str1_pro,-1)
                      paste(str2_pro, collapse = "_")
                    }
)
color_df=data.frame(read.table('tissue_color.txt',sep='\t',header = 1,quote='',fill=T,comment.char =''))
tissue_all=color_df$tissue
color_all=color_df$tissue_cor_v2
names(color_all)=tissue_all

tissue_color_list=list(Tissue=color_all)
tissue_df=data.frame(unlist(tissue_anno2))
colnames(tissue_df)='Tissue'
tissue_anno = HeatmapAnnotation(df =tissue_df, col = tissue_color_list)
plot(tissue_anno)
gene_all=lapply(tissue_anno1, 
                function(x) {
                  tail(unlist(strsplit(x, "_")),n=1)
                })

gene_all_sort=unlist(gene_all)
get_first_letter <- function(string) {
  substr(string, 1, 1)
}
gene_all_sort_pro <- sort(gene_all_sort)
gene_all_unique=unique(unlist(gene_all_sort_pro))
color_all_gene= colorRampPalette(c('#E3475A','#FFFFFF','#626EB3'))(n = length(gene_all_unique))
names(color_all_gene)=gene_all_unique

#############设置颜色#######
anno_df = data.frame(Tissue = tissue_df$Tissue,
                     Gene = unlist(gene_all))

col_fun= colorRamp2(c(0,-log10(0.05),40),c('#80AACA','#FFFFFF','#EE7B6C'))
colnames(enrich_p_value_log10)=lapply(tissue_anno1, 
                                      function(x) {
                                        tail(unlist(strsplit(x, "_")),n=1)
                                      })

tissue_gene_annot = HeatmapAnnotation(df = anno_df,
                                      col = list(Tissue = color_all,
                                                 Gene =color_all_gene
                                      ),show_legend =FALSE
)


pdf('share_pathway_pvalue_nolegend.pdf',width=8, height=(11/3))
pushViewport(viewport(gp = gpar(fontfamily = "sans")))
Heatmap_p=Heatmap(enrich_p_value_log10,name = "-log10(p value)",col=col_fun,show_column_dend = FALSE, show_row_dend = FALSE
                  ,heatmap_legend_param = list(legend_direction = "horizontal",title_gp  = gpar(fontsize = 8), labels_gp = gpar(fontsize = 8))
                  ,row_names_gp = gpar(fontsize = 8),column_names_gp  = gpar(fontsize = 8),  top_annotation  = tissue_gene_annot,show_column_names=F
)
draw(Heatmap_p, newpage = FALSE)
popViewport()
dev.off()


In [ ]:
## share pathway subnetwork number
top20_pathway_num=pd.read_csv('/top20_pathway_share_30_v3.txt',sep='\t')
all_pathway_num=pd.read_csv('/Reactome_enrich_subnetwork_counts.txt',sep='\t')

fig, axes = plt.subplots(1,1,figsize=(1.5,4.4))
plt.rcParams['axes.linewidth'] = 0.5
sns.barplot(data=top20_pathway_num,x='enrich_counts',y='Pathway_name',color='#d83b4e',ax=axes)
axes.spines['top'].set_visible(False)
axes.spines['right'].set_visible(False)
axes.set_xlabel('Number of significantly enriched subnetworks',fontsize=8, fontname='Arial')
axes.set_ylabel('',fontsize=8, fontname='Arial')
axes.tick_params(axis='x', labelsize=8, width=0.5)
axes.tick_params(axis='y', labelsize=8, width=0.5)
plt.savefig('/share_enrich_pathway_subnetwork_number_30_v6.pdf',dpi=300,bbox_inches='tight')


all_pathway_num['Pathway_name']=all_pathway_num['Pathway_name'].str.title()
all_pathway_num_zero=all_pathway_num[all_pathway_num['enrich_counts']!=0]
all_pathway_num_zero['enrich_subnetwork_all']=all_pathway_num_zero['enrich_subnetwork_all'].str.replace('Primary_','')
tissue_color=pd.read_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/donor_number_circos/tissue_color.txt',sep='\t')

pathway_count_all=pd.DataFrame()
for pos in range(0,all_pathway_num_zero.shape[0]):
    all_pathway_num_zero_eg=all_pathway_num_zero.iloc[pos:(pos+1):]
    tissue_eg=all_pathway_num_zero_eg['enrich_subnetwork_all'].str.split('|').iloc[0]
    tissue_eg_pro=pd.Series(tissue_eg).str.split('_').str[::-1].str[1::].str[::-1].str.join('_')
    gene_eg=pd.Series(tissue_eg).str.split('_').str[-1]
    gene_eg_df=pd.DataFrame(gene_eg)
    tissue_eg_pro_df=pd.DataFrame(tissue_eg_pro)
    tissue_eg_pro_df.columns=['tissue']
    tissue_eg_pro_df_rename=pd.merge(tissue_eg_pro_df,tissue_color,on='tissue')
    gene_eg_df.reset_index(drop=True,inplace=True)
    tissue_eg_pro_df_rename.reset_index(drop=True,inplace=True)
    tissue_eg_pro_df_rename_gene=pd.concat([tissue_eg_pro_df_rename,gene_eg_df],axis=1)
    final_tissue_gene=tissue_eg_pro_df_rename_gene['tissue_name']+'-'+tissue_eg_pro_df_rename_gene[0]
    final_tissue_gene_paste='|'.join(final_tissue_gene)
    all_pathway_num_zero_eg['enrich_subnetwork_all']=final_tissue_gene_paste
    all_pathway_num_zero_eg.columns=['Pathway name','Enriched tissue name','The number enriched tissues',]
    pathway_count_all=pd.concat([all_pathway_num_zero_eg,pathway_count_all])
pathway_count_all.to_excel('/Reactome_enrich_subnetwork_counts.xlsx',index=False)



In [ ]:
## specific pathway
library(clusterProfiler)
library(ggplot2)
specific_pathway_p_GR_all=data.frame(read.table('specific_pathway_p_GR_all.txt',sep='\t',header=TRUE))

specific_pathway_p_GR_all=merge(specific_pathway_p_GR_all,tissue_color,by='tissue')
specific_pathway_p_GR_all$pathway_id_plot <- factor(specific_pathway_p_GR_all$pathway_id_pro, levels = unique(specific_pathway_p_GR_all$pathway_id_pro))
specific_pathway_p_GR_all$Tissue_gene=paste(specific_pathway_p_GR_all$tissue_name,specific_pathway_p_GR_all$seed_gene,sep=' ')
specific_pathway_p_GR_all_sort=specific_pathway_p_GR_all[order(specific_pathway_p_GR_all$Tissue_gene),]

pdf('G:\\课题\\tissue_specificity_manuscript\\素材\\specific_enrich_pathway\\specific_pathway_plot_v2.pdf',height = 4, width = 8/2)
ggplot(specific_pathway_p_GR_all_sort, aes(Tissue_gene,pathway_id_plot ), showCategory=1) +
  geom_point(aes(color=FDR_BH , size=GeneRatio))+theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 8),
                                                       axis.line.y = element_line(size = (0.5/1.07)*0.5),
                                                       axis.line.x = element_line(size = (0.5/1.07)*0.5) ,
                                                       axis.ticks.y = element_line(size = (0.5/1.07)*0.5) ,
                                                       axis.ticks.x = element_line(size = (0.5/1.07)*0.5) ,
                                                       axis.text.y = element_text(size = 8)  ,
                                                       axis.title.x = element_blank()  ,axis.title.y = element_blank(),
                                                       panel.border = element_rect(color = "black", fill = NA, size = 0.5),
                                                       panel.background = element_blank())+ 
  scale_color_gradient(low = "#E3475A", high = "#626EB3",limits = c(0, 0.05))+scale_size_continuous(range = c(0.5, 3))+
  theme(legend.position = "none") +theme(panel.grid.minor = element_blank(), panel.grid.major = element_blank())
dev.off()

pdf('C:\\Users\\tianyi\\Desktop\\coding\\subnetwork_number_heatmap\\specific_pathway_plot_legend.pdf',height = 4, width = 8)
ggplot(specific_pathway_p_GR_all_sort, aes(Tissue_gene,pathway_id_plot ), showCategory=1) +
  geom_point(aes(color=FDR_BH , size=GeneRatio))+theme(axis.text.x = element_text(angle = 90, hjust = 1, size = 8),
                                                       axis.text.y = element_text(size = 8)  ,axis.title.x = element_blank()  ,axis.title.y = element_blank(),
                                                       panel.border = element_rect(color = "black", fill = NA, size = 0.5),
                                                       panel.background = element_blank())+ 
  scale_color_gradient(low = "#E3475A", high = "#626EB3",limits = c(0, 0.05))+scale_size_continuous(range = c(0.5, 3))+
  theme(panel.grid.minor = element_blank(), panel.grid.major = element_blank())+theme(legend.position = "bottom")
dev.off()


In [ ]:
## ## gene number in subnetwork重新计算
exp_gini_score=pd.read_csv('coding_exp_gini_score_median_v2.txt',sep='\t')
driver_gini=pd.read_csv('gini_count4_driver_tissue_v2.txt',sep='\t')
background_network=load_network_from_file('human_ppi_v03.tsv')
cancer_gene_type=pd.read_csv('cosmic_driver_somatic_tissue_v2.txt',sep='\t')
primary_subnetwork=pd.read_csv('primary_subnetwork_all.txt',sep='\t',header=0)
primary_subnetwork.loc[((primary_subnetwork['seed']=='APC') & (primary_subnetwork['tissue']=='Bowel')),:].shape
ts_gene_all=exp_gini_score[exp_gini_score['gini']>0.8]
ts_gene_all_class=ts_gene_all[['tissue','gene_id']].drop_duplicates()
ts_gene_all_class['ts_exp_class']='ts_exp'
ts_mut_all=driver_gini[driver_gini['Gini']>0.8]
ts_mut_all['gene_id']=ts_mut_all['Driver_Hgvsp'].str.split(':p.').str[0].tolist()
ts_mut_all_class=ts_mut_all[['Tissue','gene_id']].drop_duplicates()
ts_mut_all_class['ts_mut_class']='ts_mut'
cancer_gene_type['driver_class']='driver'
cancer_gene_type_class=cancer_gene_type[['Gene Symbol','driver_class']].drop_duplicates()

gene_class1=pd.merge(primary_subnetwork,ts_gene_all_class,left_on='sub_nodes',right_on='gene_id',how='left')
gene_class2=pd.merge(gene_class1,ts_mut_all_class,left_on='sub_nodes',right_on='gene_id',how='left')
gene_class3=pd.merge(gene_class2,cancer_gene_type_class,left_on='sub_nodes',right_on='Gene Symbol',how='left')
gene_class4=gene_class3[['sub_nodes' , 'seed'  , 'tissue_x' ,'ts_exp_class' ,'ts_mut_class' ,'driver_class']].drop_duplicates()
gene_class4.rename(columns={'tissue_x':'tissue'},inplace=True)

gene_class4['classcification']='Subnetwork member genes'
gene_class5=gene_class4.copy()
gene_class5.loc[( (gene_class5['ts_exp_class'].notna()) & (gene_class5['ts_mut_class'].notna()) & (gene_class5['driver_class'].notna())),'classcification']='Tissue-specific genetic mutations & expressed genes & driver genes'
gene_class5.loc[( (gene_class5['ts_exp_class'].notna()) & (gene_class5['ts_mut_class'].notna()) & (gene_class5['driver_class'].isna())),'classcification']='Tissue-specific genetic mutations & expressed genes'
gene_class5.loc[( (gene_class5['ts_exp_class'].isna()) & (gene_class5['ts_mut_class'].notna()) & (gene_class5['driver_class'].notna())),'classcification']='Tissue-specific genetic mutations & driver genes'
gene_class5.loc[( (gene_class5['ts_exp_class'].notna()) & (gene_class5['ts_mut_class'].isna()) & (gene_class5['driver_class'].notna())),'classcification']='Tissue-specific expressed genes & driver genes'
gene_class5.loc[( (gene_class5['ts_exp_class'].notna()) & (gene_class5['ts_mut_class'].isna()) & (gene_class5['driver_class'].isna())),'classcification']='Only tissue-specific genetic mutations'
gene_class5.loc[( (gene_class5['ts_exp_class'].isna()) & (gene_class5['ts_mut_class'].notna()) & (gene_class5['driver_class'].isna())),'classcification']='Only tissue-specific expressed genes'
gene_class5.loc[( (gene_class5['ts_exp_class'].isna()) & (gene_class5['ts_mut_class'].isna()) & (gene_class5['driver_class'].notna())),'classcification']='Only driver genes'
gene_class5.loc[( (gene_class5['ts_exp_class'].isna()) & (gene_class5['ts_mut_class'].isna()) & (gene_class5['driver_class'].isna())),'classcification']='Subnetwork member genes'
gene_class5.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/Subnetwork_member_genes_v2.txt',sep='\t',index=False)

gene_class5.loc[( (gene_class5['ts_exp_class'].isna()) & (gene_class5['ts_mut_class'].isna()) & (gene_class5['driver_class'].notna())),:]['classcification']='Only driver genes'
all_count_data=pd.DataFrame(gene_class5[['tissue','seed','classcification']].groupby(['tissue','seed']).value_counts(['classcification']))
all_count_data['tissue']=pd.Series(all_count_data.index.tolist()).str[0].tolist()
all_count_data['seed']=pd.Series(all_count_data.index.tolist()).str[1].tolist()
all_count_data['classcification']=pd.Series(all_count_data.index.tolist()).str[2].tolist()
all_count_data.reset_index(drop=True,inplace=True)

subnet_id=gene_class5[['tissue','seed']].drop_duplicates()
node_count_df_stas_all=pd.DataFrame()
for pos in range(0,subnet_id.shape[0]):
    subnet_id_eg=subnet_id.iloc[pos:(pos+1),:]
    tissue_eg=subnet_id_eg.iloc[0,0]
    seed_eg=subnet_id_eg.iloc[0,1]
    all_class=['Tissue-specific genetic mutations & expressed genes',
    'Tissue-specific genetic mutations & driver genes',
    'Tissue-specific expressed genes & driver genes',
    'Tissue-specific genetic mutations & expressed genes & driver genes',
    'Only tissue-specific genetic mutations',
    'Only tissue-specific expressed genes',
    'Only driver genes',
    'Subnetwork member genes']
    node_count_df_eg=pd.DataFrame()
    node_count_df_eg['classcification']=all_class
    node_count_df_eg['tissue']=tissue_eg
    node_count_df_eg['seed']=seed_eg
    node_count_df_eg_stas=pd.merge(node_count_df_eg,all_count_data,on=['tissue','seed','classcification'],how='left')
    node_count_df_eg_stas.fillna(0,inplace=True)
    node_count_df_stas_all=pd.concat([node_count_df_eg_stas,node_count_df_stas_all])
node_count_df_stas_all.to_csv('primary_subnetwork_gene_count_all_subnetwork_level_barplot_check_v3.txt',sep='\t',index=False)

In [ ]:
## gene number in subnetwork
library(dplyr)
library(paletteer)
library(ggplot2)
library(ggpubr)
library("ggsci")
library(stringr)


primary_ts_subnetwork=data.frame(read_excel('TS_subnetwork_26_4_12.xlsx' ))
table(primary_ts_subnetwork $ Gene.reclassification)
primary_ts_subnetwork$subnetwork_id=paste(primary_ts_subnetwork$'Tissue' , primary_ts_subnetwork$'Subnetwork.seed.genes',sep='-')
primary_ts_subnetwork_member_number=data.frame(table(primary_ts_subnetwork$subnetwork_id))
colnames(primary_ts_subnetwork_member_number)=c('subnetwork_id','total_gene_number')
primary_ts_subnetwork_classi_number=primary_ts_subnetwork %>%
  group_by(Tissue ,Subnetwork.seed.genes) %>%
  count(  Gene.reclassification, name = "gene_number")
primary_ts_subnetwork_classi_number$subnetwork_id=paste(primary_ts_subnetwork_classi_number$'Tissue' , primary_ts_subnetwork_classi_number$'Subnetwork.seed.genes',sep='-')
primary_ts_subnetwork_member_number=primary_ts_subnetwork_member_number[order(primary_ts_subnetwork_member_number$total_gene_number,decreasing = F),]
primary_ts_subnetwork_classi_number$subnetwork_id=factor(primary_ts_subnetwork_classi_number$subnetwork_id, levels = unique(primary_ts_subnetwork_member_number$subnetwork_id))
primary_ts_subnetwork_classi_number$Gene.reclassification= gsub(
  "Driver|driver",
  "cancer",
  primary_ts_subnetwork_classi_number$Gene.reclassification
)
table(  primary_ts_subnetwork_classi_number$Gene.reclassification
)
group_colors= c("cancer genes" = "#A8C89D", 
                "Subnetwork member genes" = "#DFC379",
                "TSEGs" = "#CB3B4B",
                "TSMGs&cancer genes" = "#EF9BBB",
                "TSMGs"= "#3580B2",
                "TSEGs&cancer genes"= "#7FC165",
                "TSMGs&TSEGs"= "#CFE6ED",
                "TSMGs&TSEGs&cancer genes"= "#9395C9")  # 请根据实际情况设置颜色
library(ggpubr)
pdf('sunbnetwork_number_stack_barplot_nolegend_position_sort_v6.pdf',width=8, height=(89.562/58.518)*((11/3)*(2/3)))
ggplot(data = primary_ts_subnetwork_classi_number,aes(x=gene_number,y=subnetwork_id,group=Gene.reclassification ))+
  geom_bar(stat = "identity" , position="stack", width =1 ,aes(fill=Gene.reclassification) ) +  theme_minimal()+
  scale_fill_manual(values = group_colors)+ ##用于手动设置离散型填充颜色的函数
  theme(legend.position = "none",    axis.ticks.y = element_blank(),  axis.text.y = element_blank(),
        axis.line = element_line(linewidth = 0.703125 ),
        panel.grid=element_blank(),
        axis.line.y = element_line(linewidth = (0.5/1.07)*0.5),
        axis.line.x = element_line(linewidth = (0.5/1.07)*0.5) ,
        axis.ticks.x = element_line(linewidth = (0.5/1.07)*0.5) ,
        axis.text.x = element_text(size = 8, family = "sans", 
                                   color = "black")) + # 去除 y 轴刻度线
  xlab('Subnetwork genes number')+ylab('Subnetwork')+
  font("xlab",size = 8, family = "sans", color = "black")+
  font("ylab",size = 8, family = "sans", color = "black")+
  scale_y_discrete(expand = c(0, 0)) +  # 确保 y 轴从原点开始
  scale_x_continuous(expand = c(0, 0), limits = c(0, max(primary_ts_subnetwork_classi_number$gene_number) + 15))
dev.off()


subnetwork_number=data.frame(read.table('primary_subnetwork_gene_count_all_subnetwork_level_barplot_v2.txt',sep='\t',header=1))
subnetwork_number_type=subnetwork_number[which(subnetwork_number$classcification=='Tissue-specific genetic mutations & expressed genes & driver genes'),]
subnetwork_number_type_sort=subnetwork_number_type[order(subnetwork_number_type$gene_number,decreasing=T),]
gini_index_data=data.frame(read.table('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/driver_mut_distribution_gini/driver_cancer_max_prevalance_plot_all_color.txt',sep='\t',header=1))
primary_subnetwork=data.frame(read.table('/h/tianyi/TS_datasets_reversion/network/OncoNiche/primary_subnetwork_all.txt',sep='\t',header=1))
node_counts=data.frame(table(primary_subnetwork$sub_nodes))
node_counts=node_counts[order(node_counts$Freq,decreasing=T),]
gini_index_data_sort=gini_index_data[order(gini_index_data$max_log10,decreasing=T),]
gene_split=strsplit(as.character(gini_index_data_sort$driver_mut),':p.')
                                 
gene_split_df <- data.frame(gene_split)
gini_index_data_sort$gene=as.character(unlist(gene_split_df[1,]))
gini_index_data_sort_top15=unique(gini_index_data_sort[,c('max_tissue','gene')])

subnetwork_number_type_sort_tissuegene=subnetwork_number_type_sort[,c('tissue','seed')]
gini_index_data_sort_top15_tissuegene=gini_index_data_sort_top15[,c('max_tissue','gene')]
colnames(subnetwork_number_type_sort_tissuegene)=c('tissue','gene')
colnames(gini_index_data_sort_top15_tissuegene)=c('tissue','gene')

node_counts_bscore=merge(gini_index_data_sort,node_counts,by.x='gene',by.y='Var1')
write.table(node_counts_gini,'node_counts_bscore.txt',sep='\t',row.names=F)
filter_tissue_gene=unique(rbind(subnetwork_number_type_sort_tissuegene,gini_index_data_sort_top15_tissuegene))

##############在20个子网络中，每个挑选出4个基因。方法1. 类型大于等于两个 2. betweenness最大

###读取betweenness
seed_eg='APC'
tissue_eg='Bowel'

def load_network_from_file(network_file):
  fh = open(network_file, "rb")
  G = nx.read_edgelist(fh)
  fh.close()
  return G
background_network=nx.Graph(load_network_from_file('/h/tianyi/TS_datasets_reversion/network/IAS_cancer_network/human_ppi_v03.tsv'))

all_betweenness_all=pd.DataFrame()
for pos in range(0,primary_onconiche_subnetwork_seed.shape[0]):
  primary_onconiche_subnetwork_seed_eg=primary_onconiche_subnetwork_seed.iloc[pos:(pos+1),:]
  seed_eg=''.join(primary_onconiche_subnetwork_seed_eg['seed'])
  tissue_eg=''.join(primary_onconiche_subnetwork_seed_eg['tissue'])
  print(tissue_eg)
  input_data=f'/h/tianyi/TS_datasets_reversion/primary/predict_drug/different_methods_core_nodes_auc/Primary_OncoNiche/{tissue_eg}/{tissue_eg}_{seed_eg}_betweenness_centrality_result.txt'
  try:
    betweenness_centrality_result=pd.read_csv(input_data,sep='\t')
    all_betweenness_all=pd.concat([betweenness_centrality_result,all_betweenness_all])
  except:
    pass


all_betweenness_all.to_excel('/h/tianyi/TS_datasets_reversion/primary/predict_drug/different_methods_core_nodes_auc/Primary_OncoNiche/all_betweenness_all.xlsx',index=False)
###基因分类
primary_ts_mut_exp=pd.read_csv('/primary_ts_mut_exp.txt',sep='\t')
primary_onconiche_subnetwork=pd.read_csv('/primary_subnetwork_all.txt',sep='\t')
driver_mut_type=pd.read_csv('/cosmic_driver_somatic_tissue_v2.txt',sep='\t')[['Gene Symbol','Role in Cancer']].drop_duplicates()
primary_onconiche_subnetwork_seed=primary_onconiche_subnetwork[['seed','tissue']].drop_duplicates()

all_classification_df_all=pd.DataFrame()
for pos in range(0,primary_onconiche_subnetwork_seed.shape[0]):
    primary_onconiche_subnetwork_seed_eg=primary_onconiche_subnetwork_seed.iloc[pos:(pos+1),:]
    seed_eg=''.join(primary_onconiche_subnetwork_seed_eg['seed'])
    tissue_eg=''.join(primary_onconiche_subnetwork_seed_eg['tissue'])
    exp_gini_ts_tissue_eg=pd.read_csv(f'/h/tianyi/TS_datasets_reversion/primary/FDRnet/detected_subnetwork_methods/uKIN-master/input/Global_Gini_CGC_driver_v2/{tissue_eg}_global_Gini/tissue_exp_score',sep='\t',header=None)
    mut_gini_ts_tissue_eg=pd.read_csv(f'/h/tianyi/TS_datasets_reversion/primary/FDRnet/detected_subnetwork_methods/uKIN-master/input/Global_Gini_CGC_driver_v2/{tissue_eg}_global_Gini/tissue_mut_score',sep='\t',header=None)
    subnet_gene_eg=primary_onconiche_subnetwork[ (primary_onconiche_subnetwork['seed']==seed_eg) & (primary_onconiche_subnetwork['tissue']==tissue_eg)]
    subnet_gene_eg_pro=pd.DataFrame(np.union1d(subnet_gene_eg['sub_nodes'],subnet_gene_eg['seed']))
    subnet_gene_eg_pro.columns=['Subnetwork_members']
    subnet_gene_eg_pro['TS_mut_class']=''
    subnet_gene_eg_pro['TS_exp_class']=''
    ts_mut_exp_eg=primary_ts_mut_exp[   (primary_ts_mut_exp['tissue']==tissue_eg) & (primary_ts_mut_exp['seed']==seed_eg)]
    ts_exp_genes_eg=np.intersect1d(exp_gini_ts_tissue_eg[1],subnet_gene_eg['sub_nodes'])
    ts_mut_genes_eg=np.intersect1d(mut_gini_ts_tissue_eg[1],subnet_gene_eg['sub_nodes'])
    driver_subnet=np.intersect1d(subnet_gene_eg['sub_nodes'],driver_mut_type['Gene Symbol'])
    M_E=np.intersect1d(ts_mut_genes_eg,ts_exp_genes_eg)
    M_D=np.intersect1d(ts_mut_genes_eg,driver_mut_type['Gene Symbol'])
    E_D=np.intersect1d(ts_exp_genes_eg,driver_mut_type['Gene Symbol'])
    M_E_D1=np.intersect1d(ts_mut_genes_eg,ts_exp_genes_eg)
    M_E_D2=np.intersect1d(M_E_D1,driver_subnet)
    M_E=np.setdiff1d(M_E,M_E_D2)
    M_D=np.setdiff1d(M_D,M_E_D2)
    E_D=np.setdiff1d(E_D,M_E_D2)
    only_M1=np.union1d(driver_subnet,ts_exp_genes_eg).tolist()+['No info']
    only_M2=np.setdiff1d(ts_mut_genes_eg,only_M1)
    only_E1=np.union1d(ts_mut_genes_eg,driver_subnet).tolist()+['No info']
    only_E2=np.setdiff1d(ts_exp_genes_eg,only_E1)
    only_D1=np.union1d(ts_mut_genes_eg,ts_exp_genes_eg).tolist()+['No info']
    only_D2=np.setdiff1d(driver_subnet,only_D1)
    others1=list(ts_mut_genes_eg)+list(ts_exp_genes_eg)+driver_subnet.tolist()
    others2=np.setdiff1d(subnet_gene_eg['sub_nodes'],others1)
    M_E_D_df=pd.DataFrame()
    M_E_D_df['gene']=M_E_D2
    M_E_D_df['classification']='Tissue-specific genetic mutations & expressed genes & driver genes'
    M_E_df=pd.DataFrame()
    M_E_df['gene']=M_E
    M_E_df['classification']='Tissue-specific genetic mutations & expressed genes'
    M_D_df=pd.DataFrame()
    M_D_df['gene']=M_D
    M_D_df['classification']='Tissue-specific genetic mutations & driver genes'
    E_D_df=pd.DataFrame()
    E_D_df['gene']=E_D
    E_D_df['classification']='Tissue-specific expressed genes & driver genes'
    all_classification_df=pd.concat([M_E_D_df,M_E_df,M_D_df,E_D_df])
    all_classification_df['tissue']=tissue_eg
    all_classification_df['seed']=seed_eg
    all_classification_df_all=pd.concat([all_classification_df,all_classification_df_all])
all_classification_df_all.to_excel('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/gene_classification_insubnetwork.xlsx',index=False)

all_classification_df_all=pd.read_excel('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/gene_classification_insubnetwork.xlsx')
all_betweenness_all=pd.read_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/all_seed_tissue_betweenness.txt',sep='\t')
betweenness_classification_all=pd.merge(all_classification_df_all,all_betweenness_all,on=['gene','tissue' , 'seed'])

betweenness_classification_max_all=pd.DataFrame()
for pos in range(0,primary_onconiche_subnetwork_seed.shape[0]):
  primary_onconiche_subnetwork_seed_eg=primary_onconiche_subnetwork_seed.iloc[pos:(pos+1),:]
  seed_eg=''.join(primary_onconiche_subnetwork_seed_eg['seed'])
  tissue_eg=''.join(primary_onconiche_subnetwork_seed_eg['tissue'])
  betweenness_classification_eg=betweenness_classification_all[ (betweenness_classification_all['seed']==seed_eg) & 
                                                                (betweenness_classification_all['tissue']==tissue_eg)]
  betweenness_classification_max_eg=betweenness_classification_eg.groupby('classification').max()
  betweenness_classification_max_eg['classification']=betweenness_classification_max_eg.index.tolist()
  betweenness_classification_max_eg.reset_index(drop=True,inplace=True)
  ###TS seed betweenness
  all_betweenness_eg=all_betweenness_all[ (all_betweenness_all['seed']==seed_eg) &
                                          (all_betweenness_all['tissue']==tissue_eg)&
                                          (all_betweenness_all['gene']==seed_eg)]
  all_betweenness_eg=all_betweenness_eg[['gene', 'tissue' ,'seed' , 'betweenness_centrality']]
  all_betweenness_eg['classification']='TS_seed'
  betweenness_classification_max_addseed_eg=pd.concat([betweenness_classification_max_eg,all_betweenness_eg])
  betweenness_classification_max_all=pd.concat([betweenness_classification_max_addseed_eg,betweenness_classification_max_all])
  betweenness_classification_max_all=betweenness_classification_max_all[betweenness_classification_max_all['betweenness_centrality']!=0]

betweenness_classification_max_all.to_excel('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/betweenness_classification_group_max_v2.xlsx' ,index=False)

betweenness_classification_max_all=pd.read_excel('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/betweenness_classification_group_max_v2.xlsx')

betweenness_classification_max_all_v2=betweenness_classification_max_all.drop(['classification'],axis=1).groupby(['tissue'   ,'seed']).max()
betweenness_classification_max_all_v2['tissue']=pd.Series(betweenness_classification_max_all_v2.index.tolist()).str[0].tolist()
betweenness_classification_max_all_v2['seed']=pd.Series(betweenness_classification_max_all_v2.index.tolist()).str[1].tolist()
betweenness_classification_max_all_v2.reset_index(drop=True,inplace=True)
betweenness_classification_max_all_v2_sort=betweenness_classification_max_all_v2.sort_values(['betweenness_centrality'])[::-1]

betweenness_gene_matrix=betweenness_classification_max_all_v2_sort[['gene', 'tissue' ,'seed' , 'betweenness_centrality']].drop_duplicates()
tissue_all=betweenness_classification_max_all.sort_values(['tissue'])['tissue'].drop_duplicates().tolist()
gene_all=betweenness_classification_max_all_v2_sort['gene'].drop_duplicates().tolist()[0:50]
betweenness_gene_matrix_select=betweenness_gene_matrix[betweenness_gene_matrix['gene'].isin(gene_all)]

betweenness_gene_matrix_df=pd.DataFrame(index=gene_all,columns=tissue_all)
for pos_eg in list(range(0,betweenness_gene_matrix_select.shape[0])):
  betweenness_gene_matrix_eg=betweenness_gene_matrix_select.iloc[pos_eg:(pos_eg+1),:]
  gene_eg=betweenness_gene_matrix_eg['gene'].iloc[0]
  tissue_eg=betweenness_gene_matrix_eg['tissue'].iloc[0]
  betweenness_centrality_eg=betweenness_gene_matrix_eg['betweenness_centrality'].iloc[0]
  betweenness_gene_matrix_df.loc[gene_eg,tissue_eg]=betweenness_centrality_eg

betweenness_gene_matrix_df.fillna(0,inplace=True)
betweenness_gene_matrix_df.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/betweenness_gene_matrix_df_v3.txt',sep='\t')

driver_gini=pd.read_csv('/h/tianyi/TS_datasets_reversion/primary/gini_count_cancer/gini_count4_driver_tissue_v2.txt',sep='\t')
driver_gini['gene']=driver_gini['Driver_Hgvsp'].str.split(':').str[0].tolist()
driver_gini_max=pd.DataFrame()
for tissue_eg in driver_gini['Tissue'].drop_duplicates().tolist():
  driver_gini_eg=driver_gini[driver_gini['Tissue']==tissue_eg]
  gini_max_eg=round(driver_gini_eg['Gini'].max(),3)
  driver_gini_eg_max=driver_gini_eg[round(driver_gini_eg['Gini'],3)==gini_max_eg]
  driver_gini_eg_max.reset_index(drop=True,inplace=True)
  driver_gini_max=pd.concat([driver_gini_eg_max,driver_gini_max])
driver_gini_max.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/driver_gini_max.txt',sep='\t')

In [ ]:
#####计算betweenness centrality########

subnetwork_data=primary_subnetwork
tissue='Skin'
seed='BRAF'
def betweenness_centrality_fun(subnetwork_data,tissue,seed):
    try:
        pathway=subnetwork_data[(subnetwork_data['tissue']==tissue) & (subnetwork_data['seed']==seed)]
        pathway_node=np.union1d(pathway['sub_nodes'],pathway['seed'])
        G=background_network.subgraph(pathway_node)
        pagerank_result=nx.betweenness_centrality(G,normalized=False)
        pagerank_result_pro=pd.Series(pagerank_result)
        pagerank_result_pro1=pd.DataFrame()
        pagerank_result_pro1['gene']=pagerank_result_pro.index.tolist()
        pagerank_result_pro1['betweenness_centrality']=pagerank_result_pro.tolist()
        pagerank_result_pro1['tissue']=tissue
        pagerank_result_pro1['seed']=seed
        pagerank_result_seed=pagerank_result_pro1.sort_values(by='betweenness_centrality',ascending=False)
    except:
        pagerank_result_seed=pd.DataFrame()
    return(pagerank_result_seed)
primary_subnetwork=pd.read_csv('/h/tianyi/TS_datasets_reversion/network/OncoNiche/primary_subnetwork_all.txt',sep='\t')
tissue_seed_all=primary_subnetwork[['tissue','seed']].drop_duplicates()
all_seed_betweenness=pd.DataFrame()
for pos_eg in range(0,tissue_seed_all.shape[0]):
    tissue_eg=tissue_seed_all.iloc[pos_eg,0]
    seed_eg=tissue_seed_all.iloc[pos_eg,1]
    seed_betweenness_eg=betweenness_centrality_fun(primary_subnetwork,tissue_eg,seed_eg)
    print(tissue_eg,seed_eg)
    all_seed_betweenness=pd.concat([seed_betweenness_eg,all_seed_betweenness])
all_seed_betweenness.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/seed_betweenness_type_classification_heatmap/all_seed_tissue_betweenness.txt',sep='\t',index=False)

In [ ]:
## gene-tissue heatmap####
tissue_color=pd.read_csv('/tissue_color.txt',sep='\t')
tissue_color=tissue_color[['tissue','tissue_name']]
node_counts_bscore=pd.read_csv('/node_counts_bscore.txt',sep='\t')
node_count_20=node_counts_bscore[['gene','Freq']].drop_duplicates().sort_values('Freq')[::-1].iloc[0:30,:]
node_bscore_20=node_counts_bscore[['gene','max_log10']].drop_duplicates().sort_values('max_log10')[::-1].iloc[0:30,:]
seed_select=np.union1d(node_count_20['gene'],node_bscore_20['gene'])
primary_subnetwork=pd.read_csv('/h/tianyi/TS_datasets_reversion/network/OncoNiche/primary_subnetwork_all.txt',sep='\t')
def load_network_from_file(network_file):
    fh = open(network_file, "rb")
    G = nx.read_edgelist(fh)
    fh.close()
    return G
background_network=load_network_from_file('/h/tianyi/TS_datasets_reversion/network/IAS_cancer_network/human_ppi_v03.tsv')
cancer_gene_type=pd.read_csv('/h/tianyi/TS_datasets_reversion/driver_gene/process/cosmic_driver_somatic_tissue_v2.txt',sep='\t')
cancer_gene_type_pro=cancer_gene_type[['Gene Symbol','Role in Cancer']].drop_duplicates()
cancer_gene_type_pro['Role in Cancer']=cancer_gene_type_pro['Role in Cancer'].str.replace(', fusion','')

primary_context_independent=pd.read_csv('/h/tianyi/TS_datasets_reversion/metastasis/compare_primary_met/ts_seed_network_jaccard/ts_gene_classification/context_dependent/context_independent_ts_gene.txt',sep='\t')
primary_context_dependent=pd.read_csv('/h/tianyi/TS_datasets_reversion/metastasis/compare_primary_met/ts_seed_network_jaccard/ts_gene_classification/context_dependent/context_dependent_ts_gene.txt',sep='\t')
primary_single_specific=pd.read_csv('/h/tianyi/TS_datasets_reversion/metastasis/compare_primary_met/ts_seed_network_jaccard/ts_gene_classification/context_dependent/single_specific_ts_gene.txt',sep='\t')

primary_context_independent_pro=primary_context_independent[['ts_seed']]
primary_context_independent_pro['classification']='Context-independent'
primary_context_dependent_pro=primary_context_dependent[['ts_seed']]
primary_context_dependent_pro['classification']='Context-dependent'
primary_single_specific.columns=['ts_seed']
primary_single_specific['classification']='Single-specific'
seed_classification=pd.concat([primary_context_independent_pro,primary_context_dependent_pro,primary_single_specific])

cancer_gene_type_pro.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/cancer_gene_type.txt',sep='\t',index=False)
seed_classification.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_genes/seed_classification.txt',sep='\t',index=False)

subnetwork_data=primary_subnetwork
tissue='Skin'
def betweenness_centrality_fun(subnetwork_data,tissue):
    try:
      pathway=subnetwork_data[subnetwork_data['tissue']==tissue]
      pathway_node=np.union1d(pathway['sub_nodes'],pathway['seed'])
      G=background_network.subgraph(pathway_node)
      pagerank_result=nx.betweenness_centrality(G,normalized=False)
      pagerank_result_pro=pd.Series(pagerank_result)
      pagerank_result_pro1=pd.DataFrame()
      pagerank_result_pro1['gene']=pagerank_result_pro.index.tolist()
      pagerank_result_pro1['betweenness_centrality']=pagerank_result_pro.tolist()
      pagerank_result_pro1['tissue']=tissue
      pagerank_result_seed=pagerank_result_pro1[pagerank_result_pro1['gene'].isin(pathway['seed'].tolist())].sort_values('betweenness_centrality')[::-1]
    except:
        pagerank_result_seed=pd.DataFrame()
    return(pagerank_result_seed)
primary_subnetwork=pd.read_csv('/h/tianyi/TS_datasets_reversion/network/OncoNiche/primary_subnetwork_all.txt',sep='\t')
tissue_all=primary_subnetwork['tissue'].drop_duplicates().tolist()
all_seed_betweenness=pd.DataFrame()
for tissue_eg in tissue_all:
    seed_betweenness_eg=betweenness_centrality_fun(primary_subnetwork,tissue_eg)
    print(tissue_eg)
    all_seed_betweenness=pd.concat([seed_betweenness_eg,all_seed_betweenness])
all_seed_betweenness.to_csv('all_seed_betweenness.txt',sep='\t',index=False)
all_seed_betweenness[all_seed_betweenness['gene']=='TP53']
primary_subnetwork_dup=primary_subnetwork[['tissue','seed']].drop_duplicates()
primary_subnetwork_dup.to_csv('seed_tissue.txt',sep='\t',index=False)

all_seed_betweenness=pd.read_csv('all_seed_betweenness.txt',sep='\t')
primary_subnetwork_dup=pd.read_csv('seed_tissue.txt',sep='\t')

seed_select=['RET','FGFR2','TP53','PIK3CA','APC','MTOR','CDKN2A','KRAS','PTEN','BRAF','ERBB2','NRAS','EGFR','IDH1','MAPK1','MYC','AR','IDH1','EP300','MET']

seed_select=["KRAS"   , "BRAF" ,   "IDH1"  ,  "PIK3CA" , "NRAS" ,   "TP53"  ,  "CTNNB1" ,
"AKT1"  ,  "PPP2R1A", "FGFR3" ,  "PTEN"   , "EGFR"   , "SF3B1"  , "APC"    ,
 "CDKN2A" , "HRAS" ,   "PCBP1"  , "MYD88" ,  "SPOP"  ,  "BCOR"  ,  "IDH2"   ,
"FLT3"  ,  "RAF1"  ,  "ERBB3"  , "CREBBP"  ,"ERBB2"  , "EP300"  , "MDM2"   ,
"ERBB4" ,  "MYC"   ,  "RB1"  ,   "HDAC1"  , "CDKN1A" , "MAPK1"  , "FGFR2"  ,
 "SMAD3" ,  "SMAD4"  , "SMARCA4" ,"PML"  ,   "E2F1"  ,  "KAT5"   , "SIRT1"  ,
 "GSK3B"   ,"SP1"   ,  "UBE2I"  , "DAXX"   , "FGFR1" ]

seed_classification_select=seed_classification[seed_classification['ts_seed'].isin(seed_select)]
all_seed_betweenness_select=all_seed_betweenness[all_seed_betweenness['gene'].isin(seed_select)]

primary_context_independent_select=all_seed_betweenness[all_seed_betweenness['gene'].isin(seed_select)]
primary_context_independent_select=pd.merge(primary_context_independent_select,tissue_color,on='tissue')
primary_context_independent_select_df=pd.DataFrame(columns=primary_context_independent_select.sort_values('tissue_name')['tissue_name'].drop_duplicates(),index=primary_context_independent_select.sort_values('betweenness_centrality')['gene'].drop_duplicates())

for tissue_eg in primary_context_independent_select.sort_values('tissue_name')['tissue_name'].drop_duplicates():
    for gene_eg in primary_context_independent_select.sort_values('betweenness_centrality')['gene'].drop_duplicates():
        try:
            all_seed_betweenness_select_eg=primary_context_independent_select.loc[(primary_context_independent_select['gene']==gene_eg )& ( primary_context_independent_select['tissue_name']==tissue_eg),:]['betweenness_centrality'].tolist()[0]
        except:
            all_seed_betweenness_select_eg=0
        primary_context_independent_select_df.loc[gene_eg,tissue_eg]=all_seed_betweenness_select_eg




cancer_gene_type_select=cancer_gene_type_pro[cancer_gene_type_pro['Gene Symbol'].isin(primary_context_independent_select_df.index.tolist())]
cancer_gene_type_select['Role in Cancer']=cancer_gene_type_select['Role in Cancer'].str.replace(', fusion','')
seed_classification_select_sort=seed_classification_select.copy()
seed_classification_select_sort['ts_seed'] = pd.Categorical(seed_classification_select_sort['ts_seed'].tolist(), categories=primary_context_independent_select_df.index.tolist(), ordered=True)
seed_classification_select_sort = seed_classification_select_sort.sort_values('ts_seed')
np.setdiff1d(seed_select,seed_classification_select_sort.dropna()['ts_seed'])
cancer_gene_type_select['Gene Symbol'] = pd.Categorical(cancer_gene_type_select['Gene Symbol'], categories=primary_context_independent_select_df.index.tolist(), ordered=True)
cancer_gene_type_select = cancer_gene_type_select.sort_values('Gene Symbol')
intersect_gene1=np.intersect1d(primary_context_independent_select_df.index.tolist(),seed_classification_select_sort['ts_seed'])
intersect_gene2=np.intersect1d(intersect_gene1,cancer_gene_type_select['Gene Symbol'])
primary_context_independent_select_df_inter=primary_context_independent_select_df.loc[pd.Series(primary_context_independent_select_df.index.tolist()).isin(intersect_gene2).tolist(),:]
primary_context_independent_select_df_inter.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/seed_betweenness_type_classification_heatmap/seed_betweenness_heatmap_top20_v2.txt',sep='\t')
seed_classification_select_sort.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/seed_betweenness_type_classification_heatmap/seed_classification_v2.txt',sep='\t',index=False)
cancer_gene_type_select.to_csv('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/seed_betweenness_type_classification_heatmap/cancer_gene_type_v2.txt',sep='\t',index=False)


library(ComplexHeatmap)
library(colorRamp2)
library(yarrr)
library(ggplot2)
setwd('D:\\原实验室课题\\oncoNiche\\OncoNiche-Manuscript-2024-10-18\\figure_reversion\\seed_betweenness_type_classification_heatmap')
heatmap_df=as.matrix(read.table('seed_betweenness_heatmap.txt',sep='\t',header=1,row.names = 1))
heatmap_df_scaled <- t(scale(t(heatmap_df)))
seed_classification=data.frame(read.table('seed_classification.txt',sep='\t',header=1,row.names = 1))
cancer_gene_type=data.frame(read.table('cancer_gene_type.txt',sep='\t',header=1,row.names = 1))
cancer_gene_type_color <- list(
  Group = c("oncogene" = "#D83B4E", "TSG" = "#3488BE","oncogene, TSG"='#B1D5A4')
)
colnames(cancer_gene_type)=c('Group')
cancer_gene_type_row_anno <- rowAnnotation(  df = cancer_gene_type,
                              col = cancer_gene_type_color)

seed_classification_color <- list(
  Group = c("Context-independent" = "#EF9BBB", "Context-dependent" = "#9FBDE4","Single-specific"='#7BC7B3')
)
colnames(seed_classification)=c('Group')

seed_classification_row_anno <- rowAnnotation(
  Gene_classification = cancer_gene_type$Group,
  Role_in_cancer = seed_classification$Group,
  col = list(
    Gene_classification =  c("oncogene" = "#D83B4E", "TSG" = "#3488BE","oncogene, TSG"='#B1D5A4'),
    Role_in_cancer =c("Context-independent" = "#EF9BBB", "Context-dependent" = "#9FBDE4","Single-specific"='#7BC7B3')
  )
)
pdf("heatmap_seed_plot.pdf", width = 8, height = 4)
heatmap_seed_plot=Heatmap(heatmap_df_scaled,
        name = "Betweenness (z score)",
        left_annotation = seed_classification_row_anno,
        col = colorRamp2(c(min(heatmap_df_scaled), max(heatmap_df_scaled)), c( "white", "red")),
        show_row_names = TRUE,
        show_column_names = TRUE,
        cluster_rows = T,             
        cluster_columns = FALSE,         
        cluster_row_slices = FALSE,       
        cluster_column_slices = FALSE ,
        show_row_dend = FALSE,
        row_names_gp = gpar(fontfamily = "sans", fontsize = 8, col = "black"),
        column_names_gp = gpar(fontfamily = "sans", fontsize = 8, col = "black"))
draw(heatmap_seed_plot)
dev.off()

In [ ]:
## subnetwork overlap gene venn

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

os.chdir('/h/tianyi/TS_datasets_reversion/sci_paper_plot_stas/data_statistic/subnetwork_venn')
primary_subnetwork_all=pd.read_csv('/h/tianyi/TS_datasets_reversion/network/OncoNiche/primary_subnetwork_all.txt',sep='\t')
tissue='Bowel'
gene='APC'
def subnet_filter(tissue,genes):
    primary_subnetwork_eg=primary_subnetwork_all[(primary_subnetwork_all['tissue']==tissue) & 
                                                (primary_subnetwork_all['seed']==gene)]
    primary_subnetwork_genelist=np.union1d(primary_subnetwork_eg['sub_nodes'],primary_subnetwork_eg['seed'])
    return primary_subnetwork_genelist

def draw_venn2(setA, setB, labels ,
               colors ,
               alpha, figsize, fontsize,filename):
    plt.figure(figsize=figsize)
    v = venn2([set(setA), set(setB)], set_labels=labels, set_colors=colors, alpha=alpha)
    for text in v.set_labels:
        text.set_fontsize(fontsize)
    for text in v.subset_labels:
        if text:  # 有些区域可能没有文本
            text.set_fontsize(fontsize)
    v.get_patch_by_id('11').set_color('#cc3c4c')
    v.get_patch_by_id('11').set_alpha(alpha)
    plt.savefig(filename,bbox_inches='tight',dpi=300)
draw_venn2(setA, setB, labels ,
               colors ,
               alpha, figsize, fontsize,filename)

lung_egfr=subnet_filter('Lung','EGFR')
cns_egfr=subnet_filter('CNS_Brain','EGFR')
setA=lung_egfr
setB=cns_egfr
labels=['Lung specific','CNS/Brain specific','Shared']
colors=['#ADCFA0','#3481B3',
'#CC3C4C']
alpha=0.6
figsize=(1.5,1.5)
fontsize=8
filename='EGFR_venn.pdf'

primary_subnetwork_all['tissue'].drop_duplicates()
setA=subnet_filter('Bowel','APC')
setB=subnet_filter('Uterus','APC')
setC=subnet_filter('Esophagus_Stomach','APC')
primary_subnetwork_all


set_labels=['Bowel','Uterus','Esophagus/Stomach']
from matplotlib_venn import venn3

plt.figure(figsize=(1.5,1.5))
v = venn3([set(setA), set(setB), set(setC)],alpha=0.6,set_labels=set_labels)
# 调整标签字体大小
for text in v.set_labels:
    text.set_fontsize(8)
# 调整交集区域数字字体大小
for text in v.subset_labels:
    if text:
        text.set_fontsize(8)

plt.savefig('APC_venn.pdf',bbox_inches='tight',dpi=300)


In [ ]:
primary_subnet_count=pd.read_csv('top20_pathway_share_30_rename_v4.txt',sep='\t')
reactome_pathway_gene=pd.read_csv('reactome_path_process_all_id.txt',sep='\t')
entrez_id=pd.read_csv('entrez_id_symbol_all.txt',sep='\t')

reactome_pathway_gene['pathway_id']=reactome_pathway_gene['path_name'].str.upper()
primary_subnet_count['pathway_id']='REACTOME '+primary_subnet_count['pathway_rename'].str.upper()
primary_subnet_count=primary_subnet_count[primary_subnet_count['pathway_id']!='REACTOME VIRAL INFECTION PATHWAYS'].iloc[0:20,:]
np.setdiff1d(primary_subnet_count['pathway_id'],reactome_pathway_gene['pathway_id'])
all_pathway_id=primary_subnet_count['pathway_id']
jaccard_matrix=pd.DataFrame(columns=all_pathway_id,index=all_pathway_id)
path1=all_pathway_id[1]
path2=all_pathway_id[2]
def jaccard_values_fun(path1,path2):
    path_gene1=reactome_pathway_gene[reactome_pathway_gene['pathway_id']==path1]['gene_id'].str.split('/').iloc[0]
    path_gene2=reactome_pathway_gene[reactome_pathway_gene['pathway_id']==path2]['gene_id'].str.split('/').iloc[0]
    jaccard_values=len(np.intersect1d(path_gene1,path_gene2)) / len(np.union1d(path_gene1,path_gene2))
    jaccard_values_df=pd.DataFrame({'path1':path1,'path2':path2,'jaccard_values':jaccard_values},index=[0])
    return(jaccard_values,jaccard_values_df)
import itertools
path_combination_all = list(itertools.product(list(all_pathway_id), repeat=2))  # 含有两个元素的组合
len(path_combination_all)
list(itertools.product([1,2,3], repeat=2)) 
jaccard_values_df_all=pd.DataFrame()
for path_combination_eg in path_combination_all:
    path1=path_combination_eg[0]
    path2=path_combination_eg[1]
    jaccard_values_df=jaccard_values_fun(path1,path2)
    jaccard_matrix.loc[path1,path2]=jaccard_values_df[0]
    print(path_combination_eg)
jaccard_matrix.to_csv('jaccard_matrix.txt',sep='\t')
jaccard_matrix[jaccard_matrix.isna()].shape